# Phase 1: Data Extraction from Dallas ISD Check Registers

This notebook extracts transaction data from 48 months of Dallas ISD check register PDFs (Sept 2021 - Aug 2025).

## Objectives
1. Examine PDF structure and identify table format
2. Test extraction methods (pdfplumber vs tabula-py)
3. Build robust extraction pipeline
4. Consolidate all transactions into single dataset
5. Perform initial data quality checks

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
from pathlib import Path
import pdfplumber
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

In [ ]:
# Set up paths
DATA_DIR = Path('../data')
RAW_DIR = DATA_DIR / 'raw'
EXTRACTED_DIR = DATA_DIR / 'extracted'
EXTRACTED_DIR.mkdir(exist_ok=True)

# Get list of all PDF files
pdf_files = sorted(list(RAW_DIR.glob('*.pdf')))
print(f"Found {len(pdf_files)} PDF files")
print(f"Date range: {pdf_files[0].stem} to {pdf_files[-1].stem}")

## Step 1: Examine Sample PDF Structure

Let's look at a few PDFs to understand the table structure and identify the best extraction method.

In [ ]:
# Examine first PDF (Oct 2021)
sample_pdf = pdf_files[0]
print(f"Examining: {sample_pdf.name}\n")

with pdfplumber.open(sample_pdf) as pdf:
    print(f"Total pages: {len(pdf.pages)}")
    print(f"\nFirst page dimensions: {pdf.pages[0].width} x {pdf.pages[0].height}")
    
    # Extract text from first page to see structure
    first_page = pdf.pages[0]
    text = first_page.extract_text()
    print("\nFirst 2000 characters of page 1:")
    print("="*80)
    print(text[:2000])

In [ ]:
# Try extracting tables from first page
with pdfplumber.open(sample_pdf) as pdf:
    first_page = pdf.pages[0]
    
    # Extract tables
    tables = first_page.extract_tables()
    print(f"Number of tables found on first page: {len(tables)}")
    
    if tables:
        print(f"\nFirst table has {len(tables[0])} rows and {len(tables[0][0])} columns")
        print("\nFirst 10 rows of first table:")
        print("="*80)
        for i, row in enumerate(tables[0][:10]):
            print(f"Row {i}: {row}")

In [ ]:
# Let's also check a middle PDF (2023) and recent PDF (2025) to see if format is consistent
middle_pdf = [f for f in pdf_files if '2023-06' in f.name][0]
recent_pdf = [f for f in pdf_files if '2025-06' in f.name][0]

print("Checking format consistency across time periods...\n")

for pdf_file in [sample_pdf, middle_pdf, recent_pdf]:
    with pdfplumber.open(pdf_file) as pdf:
        first_page = pdf.pages[0]
        tables = first_page.extract_tables()
        
        print(f"{pdf_file.name}:")
        print(f"  Pages: {len(pdf.pages)}")
        print(f"  Tables on page 1: {len(tables)}")
        if tables:
            print(f"  Columns in first table: {len(tables[0][0])}")
            print(f"  First row: {tables[0][0]}")
        print()

## Step 2: Build Extraction Function

Based on the structure examination, we'll build a function to extract data from each PDF.

In [ ]:
def extract_transactions_from_pdf(pdf_path, debug=False):
    """
    Extract transaction data from a Dallas ISD check register PDF.
    
    Args:
        pdf_path: Path to the PDF file
        debug: If True, print debug information
    
    Returns:
        DataFrame with extracted transactions
    """
    all_rows = []
    
    with pdfplumber.open(pdf_path) as pdf:
        if debug:
            print(f"Processing {pdf_path.name}: {len(pdf.pages)} pages")
        
        for page_num, page in enumerate(pdf.pages, 1):
            # Extract tables from page
            tables = page.extract_tables()
            
            for table in tables:
                if not table or len(table) < 2:  # Skip empty tables
                    continue
                
                # The first row is usually the header
                header = table[0]
                
                # Process data rows
                for row in table[1:]:
                    if row and any(cell for cell in row if cell):  # Skip empty rows
                        all_rows.append(row)
    
    if not all_rows:
        if debug:
            print(f"  Warning: No data extracted from {pdf_path.name}")
        return pd.DataFrame()
    
    # Get header from first table of first page
    with pdfplumber.open(pdf_path) as pdf:
        first_table = pdf.pages[0].extract_tables()[0]
        header = first_table[0]
    
    # Create DataFrame
    df = pd.DataFrame(all_rows, columns=header)
    
    # Add source file info
    df['source_file'] = pdf_path.name
    df['extraction_date'] = datetime.now()
    
    if debug:
        print(f"  Extracted {len(df)} transactions")
    
    return df

## Step 3: Test Extraction on Sample PDFs

In [ ]:
# Test on first PDF
print("Testing extraction on first PDF...\n")
test_df = extract_transactions_from_pdf(pdf_files[0], debug=True)

print(f"\nDataFrame shape: {test_df.shape}")
print(f"\nColumns: {test_df.columns.tolist()}")
print(f"\nFirst few rows:")
test_df.head(10)

In [ ]:
# Check data types and missing values
print("Data types:")
print(test_df.dtypes)
print("\nMissing values:")
print(test_df.isnull().sum())
print("\nBasic statistics:")
print(test_df.describe(include='all'))

In [ ]:
# Test on a few more PDFs to ensure consistency
test_files = [pdf_files[0], pdf_files[len(pdf_files)//2], pdf_files[-1]]

print("Testing extraction on sample files across time range...\n")
for pdf_file in test_files:
    df = extract_transactions_from_pdf(pdf_file, debug=True)
    print(f"  Columns: {len(df.columns)}, Shape: {df.shape}")
    print()

## Step 4: Extract All PDFs

Now we'll process all 48 PDFs and combine them into a single dataset.

In [ ]:
# Extract all PDFs
print(f"Extracting data from {len(pdf_files)} PDFs...\n")
print("="*80)

all_transactions = []
failed_files = []

for i, pdf_file in enumerate(pdf_files, 1):
    try:
        print(f"[{i}/{len(pdf_files)}] Processing {pdf_file.name}...", end=' ')
        df = extract_transactions_from_pdf(pdf_file)
        
        if len(df) > 0:
            all_transactions.append(df)
            print(f"✓ {len(df)} transactions")
        else:
            print("✗ No data extracted")
            failed_files.append(pdf_file.name)
            
    except Exception as e:
        print(f"✗ Error: {e}")
        failed_files.append(pdf_file.name)

print("\n" + "="*80)
print(f"\nExtraction complete!")
print(f"Successfully processed: {len(all_transactions)}/{len(pdf_files)}")
if failed_files:
    print(f"Failed files: {failed_files}")

In [ ]:
# Combine all transactions into single DataFrame
print("Combining all transactions...")
combined_df = pd.concat(all_transactions, ignore_index=True)

print(f"\nTotal transactions: {len(combined_df):,}")
print(f"Date range: {combined_df['source_file'].min()} to {combined_df['source_file'].max()}")
print(f"\nDataFrame shape: {combined_df.shape}")
print(f"Columns: {combined_df.columns.tolist()}")

## Step 5: Initial Data Quality Checks

In [ ]:
# Display sample of data
print("Sample of extracted data:")
combined_df.head(20)

In [ ]:
# Check for missing values
print("Missing values by column:")
print(combined_df.isnull().sum())
print(f"\nPercentage missing:")
print((combined_df.isnull().sum() / len(combined_df) * 100).round(2))

In [ ]:
# Examine unique values in key columns
# Note: Column names will be determined after we see the actual PDF structure
print("Data info:")
combined_df.info()

In [ ]:
# Check transactions per month
transactions_by_file = combined_df.groupby('source_file').size().sort_index()
print("Transactions per month:")
print(transactions_by_file)

In [ ]:
# Basic statistics
print("Basic statistics:")
combined_df.describe(include='all')

## Step 6: Save Extracted Data

In [ ]:
# Save to CSV
output_file = EXTRACTED_DIR / 'all_transactions_raw.csv'
combined_df.to_csv(output_file, index=False)
print(f"✓ Saved {len(combined_df):,} transactions to {output_file}")

# Also save as pickle for faster loading
pickle_file = EXTRACTED_DIR / 'all_transactions_raw.pkl'
combined_df.to_pickle(pickle_file)
print(f"✓ Saved pickle file to {pickle_file}")

# Save extraction metadata
metadata = {
    'extraction_date': datetime.now().isoformat(),
    'total_pdfs': len(pdf_files),
    'successful_extractions': len(all_transactions),
    'failed_files': failed_files,
    'total_transactions': len(combined_df),
    'columns': combined_df.columns.tolist(),
    'date_range': f"{pdf_files[0].name} to {pdf_files[-1].name}"
}

import json
metadata_file = EXTRACTED_DIR / 'extraction_metadata.json'
with open(metadata_file, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"✓ Saved metadata to {metadata_file}")

## Summary

This notebook has:
1. Examined the structure of Dallas ISD check register PDFs
2. Built an extraction function using pdfplumber
3. Extracted data from all 48 monthly PDFs
4. Performed initial data quality checks
5. Saved consolidated dataset for further analysis

### Next Steps
- Move to notebook 02_vendor_categorization.ipynb
- Clean and normalize vendor names
- Perform AI-assisted vendor research and categorization